# A/B Experimentation Analysis: Pricing Strategy


This notebook evaluates the impact of a pricing experiment on user conversion, revenue, and profitability.

Customers were divided into two groups:
- Control Group → Lower discount range
- Treatment Group → Higher discount range

The goal is to determine whether higher discounts improve conversion while maintaining profitability.


## Hypothesis

- H0 (Null Hypothesis): There is no significant difference between control and treatment groups.
- H1 (Alternative Hypothesis): Higher discounts significantly impact conversion, revenue, or margin.

In [ ]:
#import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, chi2_contingency

# Load datasets
customers = pd.read_csv('customers.csv')
sessions = pd.read_csv('sessions.csv')
orders = pd.read_csv('orders.csv')

# Revenue calculation
orders['revenue'] = orders['final_price'] * orders['quantity']

## KPI Table

In [ ]:
# sessions per group
sessions_count = sessions.groupby('ab_group')['session_id'].count()

# conversions
orders_count = orders.groupby('ab_group')['order_id'].count()

conversion_rate = orders_count / sessions_count

# revenue & margin
metrics = orders.groupby('ab_group')[['revenue','margin']].sum()

kpi_table = pd.concat([sessions_count, orders_count, conversion_rate, metrics], axis=1)
kpi_table.columns = ['sessions','orders','conversion_rate','revenue','margin']

### Insights

- Treatment group shows higher/lower revenue compared to control
- Margin comparison indicates profitability impact

## Conversion Rate Comparison

In [ ]:
sessions['converted'] = sessions['session_id'].isin(orders['session_id']).astype(int)

conversion_rate = sessions.groupby('ab_group')['converted'].mean()

conversion_rate.plot(kind='bar')
plt.title('Conversion Rate by Group')
plt.show()

## Statistical testing - Conversion rate (chi-square test)

In [ ]:
contingency = pd.crosstab(sessions['ab_group'], sessions['converted'])

chi2, p, _, _ = chi2_contingency(contingency)

print("Chi-square p-value:", p)

- If p < 0.05 → significant difference in conversion
- If p > 0.05 → no significant effect

## Revenue and Margin Comparison

In [ ]:
orders.groupby('ab_group')[['revenue','margin']].mean()

## Statistical Test Revenue T-test

In [ ]:
control = orders[orders['ab_group']=='Control']['revenue']
treatment = orders[orders['ab_group']=='Treatment']['revenue']

t_stat, p_val = ttest_ind(control, treatment)

print("T-test p-value:", p_val)

## Z-test for Conversion

In [ ]:
from statsmodels.stats.proportion import proportions_ztest

success = [orders_count['Control'], orders_count['Treatment']]
nobs = [sessions_count['Control'], sessions_count['Treatment']]

z_stat, p_val = proportions_ztest(success, nobs)
print("Z-test p-value:", p_val)

## New vs Returning Customer

In [ ]:
customer_orders = orders.groupby('customer_id').size()

orders['customer_type'] = orders['customer_id'].apply(
    lambda x: 'Returning' if customer_orders[x] > 1 else 'New'
)

orders.groupby(['ab_group','customer_type'])['revenue'].sum()

### Insight 

- Discounts may impact new and returning customers differently

## Key Insights

- Treatment group shows higher/lower conversion rate
- Revenue increases/decreases with higher discounts
- Margin is significantly impacted due to discounting
- Statistical tests confirm/reject significance